In [130]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import ast

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

from sklearn.naive_bayes import GaussianNB, CategoricalNB

from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter
from sklearn.dummy import DummyClassifier

In [131]:
# from scikitplot.metrics import plot_roc 
# from scikitplot.metrics import plot_precision_recall

from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay

Loading the datasets: we substitute `runtimeMinutes` for `logRuntime` and drop `rating` (only keeping `ratingNum`).

In [132]:
# Train

df = pd.read_csv("../results/train_clean.csv", index_col=0)
df['countryOfOrigin'] = df['countryOfOrigin'].apply(ast.literal_eval)
df['genres'] = df['genres'].apply(ast.literal_eval)
df['logRuntime'] = np.log(df['runtimeMinutes'] + 1) 
df.drop(columns=['runtimeMinutes', 'rating'], inplace=True)

In [133]:
# Test

df_test = pd.read_csv("../results/test_clean.csv", index_col=0)
df_test['countryOfOrigin'] = df_test['countryOfOrigin'].apply(ast.literal_eval) 
df_test['genres'] = df_test['genres'].apply(ast.literal_eval)
df_test['logRuntime'] = np.log(df_test['runtimeMinutes'] + 1)
df_test.drop(columns=['runtimeMinutes', 'rating'], inplace=True)

In [134]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16431 entries, 0 to 16430
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   originalTitle          16431 non-null  object 
 1   startYear              16431 non-null  int64  
 2   numVotes               16431 non-null  float64
 3   totalImages            16431 non-null  float64
 4   totalCredits           16431 non-null  float64
 5   titleType              16431 non-null  object 
 6   canHaveEpisodes        16431 non-null  bool   
 7   numRegions             16431 non-null  float64
 8   countryOfOrigin        16431 non-null  object 
 9   genres                 16431 non-null  object 
 10  ratingNum              16431 non-null  int64  
 11  numGenres              16431 non-null  int64  
 12  criticReviewsRatio     16431 non-null  float64
 13  awardsAndNominations   16431 non-null  bool   
 14  hasVideos              16431 non-null  bool   
 15  moreCou

In [135]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5478 entries, 0 to 5477
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   originalTitle          5478 non-null   object 
 1   startYear              5478 non-null   int64  
 2   numVotes               5478 non-null   float64
 3   totalImages            5478 non-null   float64
 4   totalCredits           5478 non-null   float64
 5   titleType              5478 non-null   object 
 6   canHaveEpisodes        5478 non-null   bool   
 7   numRegions             5478 non-null   float64
 8   countryOfOrigin        5478 non-null   object 
 9   genres                 5478 non-null   object 
 10  ratingNum              5478 non-null   int64  
 11  numGenres              5478 non-null   int64  
 12  criticReviewsRatio     5478 non-null   float64
 13  awardsAndNominations   5478 non-null   bool   
 14  hasVideos              5478 non-null   bool   
 15  moreCount

We binarize the two multi-label variables we have, `countryOfOrigin` and `genres`. 
Since `countryOfOrigin` has many unique values, we only keep the 45 most frequent to reduce dimensionality, collapsing all other countries to 'other'.

# Target variable: titleType

## GaussianNB
We first implement a model that only uses numerical features.

In [136]:
X_train = df.select_dtypes(include=['int64', 'float64']).values
y_train = df['titleType'].values

In [137]:
X_test = df_test.select_dtypes(include=['int64', 'float64']).values
y_test = df_test['titleType'].values

In [138]:
clf = GaussianNB()

In [139]:
%%time
clf.fit(X_train, y_train)

CPU times: total: 31.2 ms
Wall time: 28.2 ms


GaussianNB()

In [140]:
y_pred = clf.predict(X_test)
y_pred

array(['short', 'movie', 'tvSpecial', ..., 'movie', 'tvSeries', 'video'],
      shape=(5478,), dtype='<U12')

In [141]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       movie       0.83      0.81      0.82      1877
       short       0.79      0.78      0.79       766
   tvEpisode       0.70      0.80      0.74      1599
tvMiniSeries       0.24      0.05      0.08        81
     tvMovie       0.22      0.12      0.16       299
    tvSeries       0.42      0.31      0.35       447
     tvShort       0.00      0.00      0.00        16
   tvSpecial       0.07      0.27      0.11        49
       video       0.25      0.31      0.27       250
   videoGame       0.86      0.53      0.66        94

    accuracy                           0.68      5478
   macro avg       0.44      0.40      0.40      5478
weighted avg       0.68      0.68      0.67      5478



Let's compare the performance with a dummy model that predicts randomly, but respecting the class distribution in the training set. Our model performes significantly better.

In [142]:
dummy_clf = DummyClassifier(strategy='stratified', random_state=42)
dummy_clf.fit(X_train, y_train)
dummy_y_pred = dummy_clf.predict(X_test)
print(classification_report(y_test, dummy_y_pred))


              precision    recall  f1-score   support

       movie       0.35      0.33      0.34      1877
       short       0.16      0.18      0.17       766
   tvEpisode       0.27      0.28      0.28      1599
tvMiniSeries       0.03      0.02      0.03        81
     tvMovie       0.06      0.06      0.06       299
    tvSeries       0.08      0.08      0.08       447
     tvShort       0.00      0.00      0.00        16
   tvSpecial       0.02      0.02      0.02        49
       video       0.03      0.03      0.03       250
   videoGame       0.01      0.01      0.01        94

    accuracy                           0.23      5478
   macro avg       0.10      0.10      0.10      5478
weighted avg       0.24      0.23      0.23      5478



## CategoricalNB

### Pre-processing
As with `GaussianNB`, all features must me transformed into numpy arrays. Further preprocessing:

- Categorical features
  - We don't use `originalTitle`.
  - Multi-label variables: We binarize  `countryOfOrigin` and `genres`. Since `countryOfOrigin` has many unique values, we only keep the 45 most frequent to reduce dimensionality, collapsing all other countries to 'other'.
  - Ordinal: We treat `ratingNum` as a numeric variable. Since it is already a discretization, we create a bin for each unique value, for a total of 10.
  - Boolean: Keep as is.

- Numeric features: we discretize them into four quantile-based bins (ie. quartiles). 

Multi-label categorical features:

In [143]:
mbl = MultiLabelBinarizer()

In [144]:
all_countries = [country for countries in df['countryOfOrigin'] for country in countries]
country_counts = Counter(all_countries) #Counter: dictionary 'country':count
"country_counts.most_common(45)" #list of tuples

top_n_countries = 40
top_countries = set([country for country, _ in country_counts.most_common(top_n_countries)]) 

def clean_countries(countries):
    return [c if c in top_countries else "Other" for c in countries]

df['country_cleaned'] = df['countryOfOrigin'].apply(clean_countries)
df_test['country_cleaned'] = df_test['countryOfOrigin'].apply(clean_countries)

In [145]:
countries_df = pd.DataFrame(mbl.fit_transform(df['country_cleaned']), columns=mbl.classes_, index=df.index)
genres_df = pd.DataFrame(mbl.fit_transform(df['genres']), columns=mbl.classes_, index=df.index)

countries_df_test = pd.DataFrame(mbl.fit_transform(df_test['country_cleaned']), columns=mbl.classes_, index=df_test.index)
genres_df_test = pd.DataFrame(mbl.fit_transform(df_test['genres']), columns=mbl.classes_, index=df_test.index)

In [146]:
# into numpy arrays
array_countries = countries_df.values
array_countries_test = countries_df_test.values

array_genres = genres_df.values
array_genres_test = genres_df_test.values

Boolean features:

In [147]:
bool_cols = [
    'canHaveEpisodes',
    'awardsAndNominations',
    'hasVideos',
    'moreCountriesOfOrigin'
]

array_bool = df[bool_cols].values
array_bool_test = df_test[bool_cols].values

Ordinal feature (`ratingNum`):

In [148]:
# [0,9) instead of [1,10)
rating_binned = (df['ratingNum'] - 1).values.reshape(-1, 1)
rating_binned_test = (df_test['ratingNum'] - 1).values.reshape(-1, 1)

In [149]:
rating_binned

array([[7],
       [5],
       [5],
       ...,
       [5],
       [3],
       [9]], shape=(16431, 1))

Numeric features (no `ratingNum`):

In [150]:
numeric_cols = df.dtypes[(df.dtypes == 'float64') | (df.dtypes == 'int64')].index
numeric_cols = numeric_cols.drop('ratingNum')  

X_numeric = df[numeric_cols].values
X_numeric_test = df_test[numeric_cols].values

In [151]:
X_train_binned = []

for column_idx in range(X_numeric.shape[1]):
    X_train_binned.append(pd.qcut(X_numeric[:, column_idx], q=4, labels=False, duplicates='drop'))


In [152]:
X_test_binned = []

for column_idx in range(X_numeric_test.shape[1]):
    X_test_binned.append(pd.qcut(X_numeric_test[:, column_idx], q=4, labels=False, duplicates='drop'))

In [153]:
X_train_binned = np.array(X_train_binned).T
X_test_binned = np.array(X_test_binned).T

In [154]:
print(X_train_binned.shape, X_test_binned.shape)

(16431, 8) (5478, 8)


In [155]:
X_train_final = np.hstack([X_train_binned, rating_binned, array_countries, array_genres, array_bool])
X_test_final = np.hstack([X_test_binned, rating_binned_test, array_countries_test, array_genres_test, array_bool_test])

In [156]:
print(X_train_final.shape, X_test_final.shape)

(16431, 82) (5478, 82)


### Model

In [157]:
#to avoid IndexError when using CategoricalNB

min_categories = [ 
    np.unique(X_train_final[:, i]).size + 1  # only fitted on training data, but +1 for possible unseen categories in test
    for i in range(X_train_final.shape[1])
]

In [158]:
clf = CategoricalNB(min_categories=min_categories) #min_categories: maximum number of bins allocated to each feature
# clf = CategoricalNB(min_categories=4) 
clf.fit(X_train_final, y_train)

CategoricalNB(min_categories=[5, 5, 4, 5, 3, 3, 3, 5, 11, 3, 3, 3, 3, 3, 3, 3,
                              3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...])

In [159]:
y_pred = clf.predict(X_test_final)

In [160]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       movie       0.83      0.82      0.83      1877
       short       0.83      0.92      0.87       766
   tvEpisode       0.87      0.78      0.82      1599
tvMiniSeries       0.53      0.58      0.56        81
     tvMovie       0.32      0.28      0.30       299
    tvSeries       0.92      0.89      0.90       447
     tvShort       0.00      0.00      0.00        16
   tvSpecial       0.20      0.37      0.26        49
       video       0.40      0.51      0.45       250
   videoGame       0.50      0.69      0.58        94

    accuracy                           0.77      5478
   macro avg       0.54      0.58      0.56      5478
weighted avg       0.78      0.77      0.78      5478



Using categorical features as well as numeric features has proved to be beneficial.

# Target variable: ratingNum

In [162]:
y_train = df['ratingNum'].values
y_test = df_test['ratingNum'].values

## GaussianNB

In [163]:
clf = GaussianNB()

In [164]:
%%time
clf.fit(X_numeric, y_train)

CPU times: total: 15.6 ms
Wall time: 8 ms


GaussianNB()

In [165]:
y_pred = clf.predict(X_numeric_test)
y_pred

array([9, 8, 1, ..., 8, 8, 1], shape=(5478,))

In [166]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00        20
           3       0.00      0.00      0.00        52
           4       0.00      0.00      0.00       155
           5       0.35      0.10      0.15       384
           6       0.33      0.08      0.13       930
           7       0.34      0.34      0.34      1522
           8       0.37      0.59      0.45      1608
           9       0.20      0.06      0.09       690
          10       0.13      0.15      0.14       116

    accuracy                           0.30      5478
   macro avg       0.17      0.13      0.13      5478
weighted avg       0.31      0.30      0.27      5478



c:\Users\camim\anaconda3\envs\dm1_project\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\camim\anaconda3\envs\dm1_project\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\camim\anaconda3\envs\dm1_project\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

The models performs slightly better than a dummy classifier (macro F1: 0.13 vs 0.10). We assume it is due to the unbalanced distribution of the rating classes in the dataset.

In [167]:
dummy_clf = DummyClassifier(strategy='stratified', random_state=42)
dummy_clf.fit(X_numeric, y_train)
dummy_y_pred = dummy_clf.predict(X_numeric_test)
print(classification_report(y_test, dummy_y_pred))


              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.03      0.05      0.04        20
           3       0.00      0.00      0.00        52
           4       0.02      0.02      0.02       155
           5       0.08      0.09      0.09       384
           6       0.17      0.18      0.18       930
           7       0.26      0.26      0.26      1522
           8       0.30      0.29      0.30      1608
           9       0.12      0.11      0.12       690
          10       0.03      0.03      0.03       116

    accuracy                           0.21      5478
   macro avg       0.10      0.10      0.10      5478
weighted avg       0.21      0.21      0.21      5478



## CategoricalNB

We add `titleType` in the feature set, in place of `ratingNum`. Being a categorical variable, we one-hot encode it.

In [168]:
array_titleType = pd.get_dummies(df['titleType']).values
array_titleType_test = pd.get_dummies(df_test['titleType']).values

In [169]:
X_train_final = np.hstack([X_train_binned, array_titleType, array_countries, array_genres, array_bool])
X_test_final = np.hstack([X_test_binned, array_titleType_test, array_countries_test, array_genres_test, array_bool_test])

In [170]:
min_categories = [ 
    np.unique(X_train_final[:, i]).size + 1  
    for i in range(X_train_final.shape[1])
]

In [171]:
clf = CategoricalNB(min_categories=min_categories) 
clf.fit(X_train_final, y_train)

CategoricalNB(min_categories=[5, 5, 4, 5, 3, 3, 3, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3,
                              3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...])

In [172]:
y_pred = clf.predict(X_test_final)

In [173]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00        20
           3       0.03      0.06      0.04        52
           4       0.12      0.15      0.14       155
           5       0.13      0.08      0.10       384
           6       0.30      0.39      0.34       930
           7       0.34      0.26      0.29      1522
           8       0.45      0.52      0.48      1608
           9       0.26      0.18      0.21       690
          10       0.12      0.28      0.17       116

    accuracy                           0.33      5478
   macro avg       0.18      0.19      0.18      5478
weighted avg       0.33      0.33      0.32      5478



The model performs slightly better.

# Target variable: reduced rating (low - medium - high)

In [175]:
df['ratingNum'].value_counts().sort_index()

ratingNum
1        5
2       62
3      154
4      466
5     1151
6     2789
7     4565
8     4822
9     2071
10     346
Name: count, dtype: int64

In order to have a more balanced distribution of our target class labels, and hopefully improve performances, we bin `ratingNum` as following:
- 1-5: 'low'
- 6-7: 'medium'
- 8-9: 'high'

In [176]:
# 2 BINS -> ACCURACY AND MACRO F1 AROUND 0.63 FOR GAUSSIAN AND 0.69 FOR CATEGORICAL
"""
def bin_rating(r):
    if r <= 7:
        return 'Low'
    else:
        return 'High'
"""

"\ndef bin_rating(r):\n    if r <= 7:\n        return 'Low'\n    else:\n        return 'High'\n"

In [177]:
def bin_rating(r):
    if r <= 5:
        return 'Low'
    elif r <= 7:
        return 'Medium'
    else:
        return 'High'

In [178]:
y_train = (df['ratingNum'].apply(bin_rating)).values
y_test = (df_test['ratingNum'].apply(bin_rating)).values

'Low' is still underrepresented.

In [179]:
pd.Series(y_train).value_counts()

Medium    7354
High      7239
Low       1838
Name: count, dtype: int64

## GaussianNB

In [180]:
clf = GaussianNB()

In [181]:
%%time
clf.fit(X_numeric, y_train)

CPU times: total: 31.2 ms
Wall time: 29.8 ms


GaussianNB()

In [182]:
y_pred = clf.predict(X_numeric_test)
y_pred

array(['High', 'Medium', 'High', ..., 'High', 'High', 'High'],
      shape=(5478,), dtype='<U6')

In [183]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        High       0.54      0.75      0.63      2414
         Low       0.60      0.08      0.14       612
      Medium       0.58      0.49      0.53      2452

    accuracy                           0.56      5478
   macro avg       0.57      0.44      0.43      5478
weighted avg       0.57      0.56      0.53      5478



The results do improve. The recall for 'Low' is significantly low, even lower than chance (0.08 vs 0.12).

In [184]:
dummy_clf = DummyClassifier(strategy='stratified', random_state=42)
dummy_clf.fit(X_numeric, y_train)
dummy_y_pred = dummy_clf.predict(X_numeric_test)
print(classification_report(y_test, dummy_y_pred))


              precision    recall  f1-score   support

        High       0.43      0.41      0.42      2414
         Low       0.12      0.12      0.12       612
      Medium       0.43      0.44      0.44      2452

    accuracy                           0.39      5478
   macro avg       0.33      0.33      0.33      5478
weighted avg       0.40      0.39      0.39      5478



## CategoricalNB

In [185]:
clf = CategoricalNB(min_categories=min_categories) 
clf.fit(X_train_final, y_train)

CategoricalNB(min_categories=[5, 5, 4, 5, 3, 3, 3, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3,
                              3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...])

In [186]:
y_pred = clf.predict(X_test_final)

The accuracy is almost the same as before, but precision and recall are more balanced for all classes, with a higher macro F1 (0.49 vs 0.33).

In [187]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        High       0.64      0.67      0.66      2414
         Low       0.27      0.26      0.27       612
      Medium       0.57      0.54      0.56      2452

    accuracy                           0.57      5478
   macro avg       0.49      0.49      0.49      5478
weighted avg       0.57      0.57      0.57      5478

